In [1]:
import pandas as pd
import numpy as np
import re
from playwright.async_api import async_playwright
import asyncio

INPUT:

Current timestamp
Selected stocks list
AUM
Number of lots
Price

In [2]:
USER_DATA_DIR = "/Users/irfan.hilman/Downloads/user_data_test_local"
p = await async_playwright().start()
context = await p.chromium.launch_persistent_context(
    USER_DATA_DIR,
    headless=False,
    args=["--disable-blink-features=AutomationControlled",
        "--start-minimized",
        "--disable-background-timer-throttling",
        "--disable-renderer-backgrounding",
        "--disable-backgrounding-occluded-windows",
        "--disable-features=IsolateOrigins,site-per-process",
        "--disable-gpu",
        "--disable-extensions",
        "--disable-sync",
        "--disable-default-apps"],
    viewport=None)
if context.pages:
    page = context.pages[0]
    await page.close()

Error occurred in event listener
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyee/asyncio.py", line 79, in _emit_run
    coro: Any = f(*args, **kwargs)
                ~^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/playwright/_impl/_page.py", line 207, in <lambda>
    lambda params: self._on_frame_detached(from_channel(params["frame"])),
                   ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/playwright/_impl/_page.py", line 279, in _on_frame_detached
    self._frames.remove(frame)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^
ValueError: list.remove(x): x not in list
Error occurred in event listener
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyee/asyncio.py", line 79, in _emit_run

In [3]:
page_1 = await context.new_page()
await page_1.goto("https://stockbit.com/stream")

<Response url='https://stockbit.com/stream' request=<Request url='https://stockbit.com/stream' method='GET'>>

In [27]:
stock = 'NANO'
action = 'Buy' # Buy or Sell
order_type = 'Limit' # Market or Limit
num_lots = "20"
expiry = 'Good For Day' # Good For Day / Good Till Cancelled
price = "32"

In [ ]:
# TRANSACTIONS ORDER
# LOOP
order_trans = pd.read_excel('buy_order_transaction_example.xlsx')

for i in order_trans['Number']:

    stock = order_trans[order_trans['Number']==i]['Stock'].values[0]
    action = order_trans[order_trans['Number']==i]['Action'].values[0]
    order_type = order_trans[order_trans['Number']==i]['Order Type'].values[0]
    num_lots = str(order_trans[order_trans['Number']==i]['Lot'].values[0])
    expiry = order_trans[order_trans['Number']==i]['Expiry'].values[0]
    price = str(order_trans[order_trans['Number']==i]['Price'].values[0])

    # Select stock
    search_stock = page_1.locator('[data-cy="top-navbar-search-input-desktop"]')
    await search_stock.click()
    await search_stock.fill(stock)
    await search_stock.press("Enter")

    # Hover to container buy/sell
    container = page_1.locator('[data-overlayscrollbars-viewport]')
    await container.hover()              # give focus to the scroll area
    await container.evaluate("""(el) => {el.scrollTop = el.scrollHeight}""")

    if action == 'Buy':
        buy_tab = page_1.get_by_role("button", name=re.compile(r"^Buy$")).nth(1)
        await buy_tab.click()
    elif action == 'Sell':
        sell_tab = page_1.get_by_role("button", name="Sell").first
        await sell_tab.click()

    if order_type == 'Market':
        mo_tab = page_1.get_by_role("button", name="Market").first
        await mo_tab.click()
        lot_input = page_1.locator('[data-cy="input-buy-lot"]')
        await lot_input.click()
        await lot_input.fill(num_lots)
        buy_button = page_1.get_by_role("button", name=re.compile(r"^Buy$")).nth(2)
        await asyncio.sleep(0.1)
        is_disabled = await buy_button.is_disabled()
        if not is_disabled:
            await buy_button.click()
            confirm = page_1.get_by_role("button", name="Confirm")
            await confirm.click()
            done_button = page_1.get_by_role("button", name="Return to Orderbook")
            await done_button.click()
    elif order_type == 'Limit':
        lo_tab = page_1.get_by_role("button", name="Limit").first
        await lo_tab.click()
        price_input = page_1.locator('[data-cy="input-buy-price"]')
        await price_input.click()
        await price_input.fill(price)
        lot_input = page_1.locator('[data-cy="input-lot"]')
        await lot_input.click()
        await lot_input.fill(num_lots)
        # Expiry
        expiry_button = page_1.locator('[data-cy="select-trigger-buy-expiry"]')
        await expiry_button.click()
        # Good For Day or Good Till Cancelled
        if expiry=='Good For Day':
            gfd = page_1.get_by_role("option", name="Good For Day")
            await gfd.click()
        elif expiry=='Good Till Cancelled':
            gtc = page_1.get_by_role("option", name="Good Till Cancelled")
            await gtc.click()
        
        buy_button = page_1.get_by_role("button", name=re.compile(r"^Buy$")).nth(2)
        await asyncio.sleep(0.1)
        is_disabled = await buy_button.is_disabled()
        if not is_disabled:
            await buy_button.click()
            confirm = page_1.get_by_role("button", name="Confirm")
            await confirm.click()
            done_button = page_1.get_by_role("button", name="Return to Orderbook")
            await done_button.click()
    await asyncio.sleep(0.5)
    


In [6]:
PIN = ['1','3','3','6','6','5']

In [16]:
# OPEN PORTFOLIO PAGE

await page_1.locator('[data-cy="navbar-portfolio"] a').first.click()

pin_input = page_1.locator('.ant-modal-body:has-text("Input Trading PIN")')

try:
    await pin_input.wait_for(state="visible", timeout=1000)  # 1s should be enough
    pin_popup_appeared = True
except:
    pin_popup_appeared = False

if pin_popup_appeared:
    # fill PIN, submit, etc.
    for p in PIN:
        await page_1.keyboard.type(p)
        await asyncio.sleep(0.1)
    await page_1.get_by_role("button", name="Submit").click()
else:
    # already logged in, went straight to portfolio page
    pass

In [5]:
# READ AUM

await page_1.locator('[data-cy="navbar-portfolio"] a').first.click()

In [ ]:
# SINGLE EXECUTION

# Select stock
search_stock = page_1.locator('[data-cy="top-navbar-search-input-desktop"]')
await search_stock.click()
await search_stock.fill(stock)
await search_stock.press("Enter")

# Hover to container buy/sell
container = page_1.locator('[data-overlayscrollbars-viewport]')
await container.hover()              # give focus to the scroll area
await container.evaluate("""(el) => {el.scrollTop = el.scrollHeight}""")

if action == 'Buy':
    buy_tab = page_1.get_by_role("button", name=re.compile(r"^Buy$")).nth(1)
    await buy_tab.click()
elif action == 'Sell':
    sell_tab = page_1.get_by_role("button", name="Sell").first
    await sell_tab.click()

if order_type == 'Market':
    mo_tab = page_1.get_by_role("button", name="Market").first
    await mo_tab.click()
    lot_input = page_1.locator('[data-cy="input-buy-lot"]')
    await lot_input.click()
    await lot_input.fill(num_lots)
    buy_button = page_1.get_by_role("button", name=re.compile(r"^Buy$")).nth(2)
    await asyncio.sleep(0.1)
    is_disabled = await buy_button.is_disabled()
    if not is_disabled:
        await buy_button.click()
        confirm = page_1.get_by_role("button", name="Confirm")
        await confirm.click()
        done_button = page_1.get_by_role("button", name="Return to Orderbook")
        await done_button.click()
elif order_type == 'Limit':
    lo_tab = page_1.get_by_role("button", name="Limit").first
    await lo_tab.click()
    price_input = page_1.locator('[data-cy="input-buy-price"]')
    await price_input.click()
    await price_input.fill(price)
    lot_input = page_1.locator('[data-cy="input-lot"]')
    await lot_input.click()
    await lot_input.fill(num_lots)
    # Expiry
    expiry_button = page_1.locator('[data-cy="select-trigger-buy-expiry"]')
    await expiry_button.click()
    # Good For Day or Good Till Cancelled
    if expiry=='Good For Day':
        gfd = page_1.get_by_role("option", name="Good For Day")
        await gfd.click()
    elif expiry=='Good Till Cancelled':
        gtc = page_1.get_by_role("option", name="Good Till Cancelled")
        await gtc.click()
    
    buy_button = page_1.get_by_role("button", name=re.compile(r"^Buy$")).nth(2)
    await asyncio.sleep(0.1)
    is_disabled = await buy_button.is_disabled()
    if not is_disabled:
        await buy_button.click()
        confirm = page_1.get_by_role("button", name="Confirm")
        await confirm.click()
        done_button = page_1.get_by_role("button", name="Return to Orderbook")
        await done_button.click()
